## Tarea 1. Diseña una base de datos Cassandra para dar servicio a las lecturas y escrituras anteriores. Argumenta tus decisiones de diseño. 

### 1. Hall of fame


El “hall of fame” de cada país, es decir, para un país concreto se muestran para cada mazmorra del juego el TOP 5 de jugadores 
más rápidos de ese país, incluyendo sus tiempos.

**NOTA**: Así es como considero que podría hacerse. Por el enunciado, es posible que dungeon_id no sea parte de la clave de partición, sino de clustering. Tener eso en cuenta. Podemos preguntar. 

Como es necesario mostrar para cada país concreto, entonces esta debe ser parte de la clave de partición. Además de eso, asumimos por el enunciado que se deberá filtrar también el el id de la mazmorra (o el nombre en caso de que sea único), por lo que este también tiene que formar parte de la clave de partición. Así, las consultas se resuelven en un sólo nodo. 

Como queremos tener un orden teniendo en cuenta el tiempo, será necesario dentro de la partición de cada país-mazmorra definir claves de clustering. Para las claves de clustering, tendremos el tiempo (ordenado de manera ascendente, de menor a mayor, ya que queremos que aparezcan primero los jugadores con los menores tiempos). Como puede ser que varios jugadores hayan hecho el mismo tiempo, añadiremos el email (que según el esquema E/R es la clave primaria del jugador) para poder garantizar unicidad de la fila (así evitamos que un tiempo igual de otro usuario sobreescriba el registro). Estos últimos serán en orden ascendente (por orden alfabético)

Aunque nos pida el TOP 5, simplemente escogemos todos y ya nos encargaremos que la aplicación lea sólamente las 5 primeras filas (usando LIMIT 5). 

Para el resto de datos que aparecen en el enunciado, tendremos que incluirlos dentro de la tabla: 

- user_name
- date
- dungeon_name

Para resumir, el esquema general de la tabla sería el siguiente: 

**Nombre de la tabla**: hall_of_fame_by_country
- **Partition Key**: (country, dungeon_id)
- **Clustering Columns**: (time_minutes ASC, email ASC)
- **Otras columnas**: 
    - user_name
    - date
    - dungeon_name

En Cassandra: 


```sql
CREATE TABLE hall_of_fame_by_country (
    country TEXT,
    dungeon_id INT,
    time_minutes INT,
    email TEXT,
    user_name TEXT,
    date TIMESTAMP,
    dungeon_name TEXT STATIC,
    PRIMARY KEY ((country, dungeon_id), time_minutes, email)
) WITH CLUSTERING ORDER BY (time_minutes ASC, email ASC);


Por supuesto, como es necesario que al introducir el país, salgan todas las mazmorras, necesitaremos una tabla adicional que nos permita obtener las mazmorras de cada país. Esta tabla sería la siguiente:

**Nombre de la tabla**: dungeons_by_country
- **Partition Key**: country
- **Clustering Columns**: dungeon_id ASC
- **Otras columnas**


```sql
CREATE TABLE dungeons_by_country (
    country TEXT,
    dungeon_id INT,

    PRIMARY KEY ((country), dungeon_id)
) WITH CLUSTERING ORDER BY (dungeon_id ASC);
```

# TODO: justificar los tipos dentro de la tabla

### 2. User Statistics

.. y las estadísticas de un jugador, que muestra 
los tiempos que ha tardado en completar una mazmorra en particular ordenados de menor a 
mayor.  

Como tiene como entradas el id del usuario y el id de la mazmorra, simplemente podemos tener como clave de partición ambos atributos para poder garantizar un acceso directo a los intentos de un usuario específico para una mazmorra en concreto. 

Al igual que en el ejemplo anterior, para la clave de clustering conviene utilizar el tiempo en minutos y en orden ascendente, ya que queremos que los mejores tiempos aparezcan antes. **NOTA**: en caso de que haya dos tiempos iguales, para poder diferenciarlos, podemos usar la fecha, aunque si no simplemente no pasa nada. 

Como también se nos pide la fecha (en caso de que sea clave de clustering no será necesario), tendremos que incluirlo dentro de la tabla. 

**Nombre de la tabla**: user_statistics_by_dungeon
- **Partition Key**: (user_id, dungeon_id)
- **Clustering Columns**: (time_minutes ASC, date ASC)
- **Otras columnas**: 

```sql
CREATE TABLE user_statistics_by_dungeon (
    email text,
    dungeon_id int,
    time_minutes int,
    date TIMESTAMP,
    PRIMARY KEY ((email, dungeon_id), time_minutes, date)
) WITH CLUSTERING ORDER BY (time_minutes ASC, date ASC);

### 3. Top Horde

 El equipo de 
“game design” quiere introducir un leaderboard para las Hordas que muestre los N jugadores 
(aún no se tiene claro cuantos) que más monstruos han matado hasta el momento durante una 
Horda en concreto. El leaderboard se debe ir actualizando en “tiempo real”, hay que tener en 
cuenta que cualquier retraso a la hora de enviar/recibir la información en este punto es crítico, 
ya que se ejecuta mientras los usuarios están en pleno gameplay. La consistencia en este 
leaderboard no es tan importante ya que el equipo de “game design” ha comprobado que, si los 
rankings bailan un poco, los jugadores se motivan más porque da la sensación de que la 
competición está más tensa. Sin embargo, los rankings deben de reflejar la realidad, si bailan 
demasiado los usuarios lo notarán y se disgustarán. Por temas de latencia, los jugadores solo 
participan en una Horda con otros jugadores del mismo país, por lo que el leaderboard de Horda 
será local por país.

Como se especifica que los jugadores sólamente participan en una Horda con otros jugadores del mismo país, entonces el país tiene que estar dentro de la clave de partición. 

Además de eso, como tienen que estar separados en Hordas en concreto, es necesario incluir también dentro de la clave de partición el id del evento. 

De esta manera, podemos leer todos los participantes de una horda directamente. 

Usaremos la K (que suponemos que indica el TOP K jugadores con la mayor cantidad de kills) posteriormente a la hora de realizar la consulta. 


Dentro de la clave de clustering, es obvio que, como estarán ordenados en función de la cantidad de kills, n_killed será parte de esta, en orden descendiente (queremos que los que más arriba estén sean los que más kills hayan hecho). Para que sea única, añadiremos el user id dentro de la clave de clustering (no podemos identificar únicamente por el número de kills), asegurando así unicidad de la clave primaria.


Hay que tener en cuenta que, como el número de kills pasará a formar parte de la clave primaria, entonces no podremos actualizar el número de kills que ha realizado el jugador porque entonces estaríamos cambiando la clave primaria. 

En este caso, cada vez que un jugador mate a un monstruo, habrá que borrar el n_killed anterior e insertar el nuevo registro, lo que hará que el ranking esté desactualizado durante unos segundos. Sin embargo, justificamos esta acción por lo que dice el enunciado: *"La consistencia en este leaderboard no es tan importante... si los rankings bailan un poco, los jugadores se motivan más"*. Tendremos que realizar estas operaciones en BATCH para reducir la carga de escrituras y borrados.


Para resumir, el esquema general de la tabla sería el siguiente: 
**Nombre de la tabla**: top_horde_by_event
- **Partition Key**: (country, event_id)
- **Clustering Columns**: (n_killed DESC, user_id ASC)
- **Otras columnas**: 
    - user_name
    - email

```sql

CREATE TABLE top_horde_by_event (
    country text,
    event_id int,
    n_killed int,
    user_name text,
    email text,
    PRIMARY KEY ((country, event_id), n_killed, email)
) WITH CLUSTERING ORDER BY (n_killed DESC, email ASC);